# Unit 4 — CrewAI + Classical ML + MCP
## AI-Based Defect Reporting System
### Agentic AI & Automation — Symbiosis International University

**Learning Objectives (CO4):**
- Build Advanced Agentic AI Workflows Using CrewAI
- Train Linear Regression, Random Forest models (sklearn)
- Perform EDA and preprocess data
- Evaluate models with R², MAE, MSE
- Components of MCP — deploy Gradio as MCP server


In [ ]:
import os, sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, r2_score, mean_absolute_error, mean_squared_error, classification_report
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries loaded')

## Part A: Load and Inspect Dataset (EDA)

In [ ]:
# Load defect dataset
df = pd.read_csv('../app/ml/defect_dataset.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())
print('\nData types:')
display(df.dtypes.to_frame('dtype'))
print('\nMissing values:')
display(df.isnull().sum().to_frame('missing'))

In [ ]:
# Exploratory Data Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Severity distribution
colors = {'Critical': '#e74c3c', 'High': '#f39c12', 'Medium': '#3498db', 'Low': '#2ecc71'}
severity_counts = df['severity'].value_counts()
axes[0,0].bar(severity_counts.index, severity_counts.values,
              color=[colors.get(s) for s in severity_counts.index])
axes[0,0].set_title('Defect Severity Distribution')
axes[0,0].set_ylabel('Count')

# 2. Component distribution
comp_counts = df['component'].value_counts()
axes[0,1].barh(comp_counts.index, comp_counts.values)
axes[0,1].set_title('Defects by Component')

# 3. Priority score by severity
for sev, color in colors.items():
    subset = df[df['severity'] == sev]['priority_score']
    if len(subset) > 0:
        axes[1,0].hist(subset, bins=8, alpha=0.6, label=sev, color=color)
axes[1,0].set_title('Priority Score by Severity')
axes[1,0].legend()
axes[1,0].set_xlabel('Priority Score')

# 4. Correlation heatmap
numeric_cols = ['user_impact', 'frequency', 'reproducibility', 'priority_score']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1,1])
axes[1,1].set_title('Feature Correlation Matrix')

plt.suptitle('Defect Dataset — Exploratory Data Analysis (EDA)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../app/ml/eda_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA complete')

## Part B: Preprocessing — Encode Categoricals, Handle Missing Values

In [ ]:
COMPONENTS = ['Authentication', 'Payment', 'Database', 'API', 'Dashboard',
              'Search', 'UI', 'Notifications', 'Reports', 'Settings', 'Other']
ERROR_TYPES = ['NullPointer', 'Crash', 'DataCorruption', 'SecurityVuln',
               'Timeout', 'ConnectionFail', 'ServerError500', 'DataLoss',
               'WrongPassword', 'RenderError', 'WrongResults', 'NotSending',
               'SlowResponse', 'LoadSlow', 'FormatError', 'AlignmentIssue',
               'SaveFail', 'WrongLabel', 'ColorIssue', 'UXConfusing', 'Other']

# Encode categorical features
df['component_enc'] = df['component'].apply(lambda x: COMPONENTS.index(x) if x in COMPONENTS else len(COMPONENTS))
df['error_type_enc'] = df['error_type'].apply(lambda x: ERROR_TYPES.index(x) if x in ERROR_TYPES else len(ERROR_TYPES))

print('✅ Categorical encoding complete')
print(df[['component', 'component_enc', 'error_type', 'error_type_enc']].head())

# Handle missing values (imputation)
print(f'\nMissing values: {df.isnull().sum().sum()} (none in this clean dataset)')

# Feature matrix and targets
features = ['component_enc', 'error_type_enc', 'user_impact', 'frequency', 'reproducibility']
X = df[features].values
y_class = df['severity'].values     # Classification target
y_reg = df['priority_score'].values # Regression target

X_train, X_test, yc_train, yc_test = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)
_, _, yr_train, yr_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

print(f'\nTraining set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')

## Part C: Train Models — Linear Regression + Random Forest

In [ ]:
# 1. Linear Regression — predict priority_score
print('📈 Linear Regression (Priority Score Prediction)')
lr = LinearRegression()
lr.fit(X_train, yr_train)
yr_pred = lr.predict(X_test)

r2 = r2_score(yr_test, yr_pred)
mae = mean_absolute_error(yr_test, yr_pred)
mse = mean_squared_error(yr_test, yr_pred)

print(f'  R² Score: {r2:.4f}')
print(f'  MAE:      {mae:.4f}')
print(f'  MSE:      {mse:.4f}')
print(f'  RMSE:     {mse**0.5:.4f}')

print()
print('🌲 Random Forest Classifier (Severity Prediction)')
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
rf.fit(X_train, yc_train)
yc_pred = rf.predict(X_test)

acc = accuracy_score(yc_test, yc_pred)
print(f'  Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print()
print('Classification Report:')
print(classification_report(yc_test, yc_pred))

In [ ]:
# Save the model and run full training script
import subprocess
result = subprocess.run(['python', '../app/ml/train_model.py'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('Stderr:', result.stderr[:500])

## Part D: CrewAI Crew — Data Analyst + ML Engineer + Report Writer

In [ ]:
from app.workflows.crewai_crew import run_crewai_analysis

print('🚀 Running CrewAI Crew Analysis')
print('Agents: Data Analyst | ML Engineer | Report Writer')
print('='*60)

result = run_crewai_analysis(
    defect_description='Database connection pool exhausted under high load',
    component='Database'
)
print(result)

## Part E: MCP Server — Deploy Gradio as MCP-Compatible Tool

In [ ]:
# Demonstrate MCP client calling the server
import json
from app.ml.severity_predictor import predict_severity

print('🔌 MCP Client Demo — Calling predict_severity tool')
print('(Simulates an AI agent discovering and calling MCP tools)\n')

# Step 1: Discover tool manifest
manifest = {
    'name': 'defect-reporting-mcp-server',
    'tools': ['analyze_defect', 'predict_severity', 'generate_report']
}
print('Step 1: Discovered MCP tools:', manifest['tools'])

# Step 2: Call predict_severity tool
print('\nStep 2: Calling predict_severity via MCP...')
result = predict_severity(
    component='Payment',
    error_type='Timeout',
    user_impact=4,
    frequency=3,
    reproducibility=4
)
print(json.dumps(result, indent=2))

print()
print('To run the full MCP server:')
print('  python app/mcp/mcp_server.py')
print('  Open http://localhost:7861')